In [ ]:
"""Additive CPU diagnostic on the completed seed-0 outputs; no training or promotion."""
import base64
from datetime import datetime, timezone
import hashlib
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import tarfile
import traceback


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main(payload64, review):
    os.environ.update({'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2',
                       'PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':''})
    work=Path('/kaggle/working/neural_cluster0_r1')
    work.mkdir()
    status={'started_utc':datetime.now(timezone.utc).isoformat(),'training':False,
            'changes_primary_gate':False,'seed0_eligible_for_cell_scorer':False}
    (work/'runtime_review.json').write_text(json.dumps(review,indent=2))
    try:
        payload=base64.b64decode(payload64,validate=True)
        if hashlib.sha256(payload).hexdigest()!=review['payload_sha256']:
            raise ValueError('Code archive hash differs')
        code=work/'code'
        expected={e['path']:e for e in review['code_allowlist']}
        with tarfile.open(fileobj=io.BytesIO(payload),mode='r:gz') as archive:
            entries=archive.getmembers()
            if len(entries)!=len(expected) or {e.name for e in entries}!=set(expected):
                raise ValueError('Code allowlist differs')
            for entry in entries:
                path=(code/entry.name).resolve()
                if not entry.isfile() or not path.is_relative_to(code.resolve()):
                    raise ValueError('Unsafe source entry')
                data=archive.extractfile(entry).read()
                if hashlib.sha256(data).hexdigest()!=expected[entry.name]['sha256']:
                    raise ValueError('Source hash differs')
                path.parent.mkdir(parents=True,exist_ok=True)
                with path.open('xb') as f:
                    f.write(data)
        all_manifests=list(Path('/kaggle/input').rglob('manifest.json'))
        datasets=[p.parent for p in all_manifests if 'vcc-rete-contesti-r2' in p.parts
                  and sha(p)==review['dataset_manifest_sha256']]
        if len(datasets)!=1:
            raise ValueError('Cannot locate exact r2 dataset')
        folds=[]
        for family in review['fold_order']:
            evidence=review['fold_evidence'][family]
            candidates=[p.parent for p in all_manifests if p.parent.name==f'C_{family}_s0'
                        and sha(p)==evidence['manifest.json']['sha256']]
            if len(candidates)!=1:
                raise ValueError(f'Completed seed0 kernel output is not uniquely mounted: {family}')
            folder=candidates[0]
            for name,record in evidence.items():
                if (folder/name).stat().st_size!=record['bytes'] or sha(folder/name)!=record['sha256']:
                    raise ValueError(f'Frozen seed0 report differs: {family}/{name}')
            folds.append(folder)
        import numpy as np
        import pandas as pd
        (work/'environment.json').write_text(json.dumps({'python':sys.version,'numpy':np.__version__,
                                                        'pandas':pd.__version__},indent=2))
        script=code/'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/cluster_pds.py'
        command=[sys.executable,str(script),'--runs',*[str(p) for p in folds],
                 '--data',str(datasets[0]),'--out',str(work/'diagnostic')]
        (work/'command.json').write_text(json.dumps(command,indent=2))
        with (work/'diagnostic.log').open('x',encoding='utf-8') as log:
            child=subprocess.run(command,stdout=log,stderr=subprocess.STDOUT,text=True)
        status['returncode']=child.returncode
        if child.returncode:
            raise RuntimeError('Additive diagnostic failed; original gate remains negative')
        status['complete']=True
    except Exception:
        status.update({'complete':False,'exception':traceback.format_exc()})
        raise
    finally:
        status['finished_utc']=datetime.now(timezone.utc).isoformat()
        (work/'completion.json').write_text(json.dumps(status,indent=2))
        print(json.dumps(status))


main('...', {'prepared_utc': '2026-09-29T19:41:50.746888+00:00', 'kernel': 'davidmaisterx/vcc-lead-neural-cluster0-r1', 'private': True, 'dataset': 'davidmaisterx/vcc-rete-contesti-r2', 'dataset_manifest_sha256': '2e59376b103dc6578b9a52e3c37c655207567ecc0375481bc5e3560e7fb8b32a', 'kernel_source': 'davidmaisterx/vcc-lead-neural-sources-r1/1', 'kernel_source_mount_confirmed': False, 'payload_sha256': '48d6edcd39975a7e6c0e53dd12c48bc0b66d0e5896fc1ba3bb55529ff05e10a7', 'code_allowlist': [{'path': 'reports/analisi/lead_scientist_2026-09-29/PROTOCOLLO_NEURALE.md', 'bytes': 7452, 'sha256': '7febed5942476f38f98af8969d7cc38f22153e419c90014ea5b24f3c05068374'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_cluster0_runner.py', 'bytes': 4255, 'sha256': '50b99cf1d14595c1667846cb7f49ddbdc058fb6b80d5ea741d6f823ba4ccf431'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/PROTOCOLLO_DIAGNOSTICA.md', 'bytes': 5916, 'sha256': '5b4242c69aa2a82ee67e2cb44aad0c92e9dcad69c277dcc10e47875dd971c0c5'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/cluster_pds.py', 'bytes': 9235, 'sha256': '4413ad2662b6315c9f08c396a766ba6d9d4fa066d22ad5661b06226eb416c1e9'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/test_cluster_pds.py', 'bytes': 3669, 'sha256': 'ab3273f9c6498102bd472432b4d5ba464e7915b10ed9f572c284fe5bd462a493'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_sources.py', 'bytes': 14657, 'sha256': 'fdfc6e7cadc153e161d9679fe0ee3c3ca6a846e71df85b328ae9a726133f55b1'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/train_neural_sources.py', 'bytes': 19483, 'sha256': '2b4e9734a0ea630010e0b83c8cbbb0c48fd1707e5356c3834ee5dbb954d55f72'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/verify_neural_runs.py', 'bytes': 5185, 'sha256': 'e90e095c25f7e4e17c0cc0e4cfad71f87b57eababf49765db9c5ec624ff0e3b4'}, {'path': 'reports/modelli/rete_contesti_2026-09-27/pool.py', 'bytes': 45527, 'sha256': '10bd001536c013be8937c0ea321e0e9b7d16416ac525104f6fe0ef678f0940ce'}], 'fold_order': ['k562', 'cd4', 'orion', 'ipsc', 'rpe1'], 'fold_evidence': {'k562': {'manifest.json': {'bytes': 1951353, 'sha256': '18619ece0b096bc430250b0b1a3b5c397044a10ddcd7863877e4ae901c014c80'}, 'per_target.csv': {'bytes': 1084837, 'sha256': 'c13399182ea4e4e3d83f062b302d8260ec51a9f5325c8bdd91133b7280884bea'}, 'summary.json': {'bytes': 3208, 'sha256': 'd047a79b00517af5b99c98141293235d26e577cdf191524dea7af343fc457949'}}, 'cd4': {'manifest.json': {'bytes': 1678000, 'sha256': '1fe91f0efc7dc02fb575813d8da87b2b27f85cc5de59a5e68363a53d8376088b'}, 'per_target.csv': {'bytes': 1123945, 'sha256': 'c72e720c71e2074287cab2d40e1305722995c99f2b6917c454d9d4662cf6f105'}, 'summary.json': {'bytes': 3207, 'sha256': 'ee18e88b45e523fcb7c0764913aee6aab5622294287317beb5d6952b845b5dbb'}}, 'orion': {'manifest.json': {'bytes': 1737712, 'sha256': 'd53b5fee273320f66e7979f85024438e4a0a2e5cd489d2c7a0beeeacd5e2878d'}, 'per_target.csv': {'bytes': 740900, 'sha256': 'bf297c37b4ac3f18e503218b2cbbe54dad605f5c83f6839ca63683e7c6bd724f'}, 'summary.json': {'bytes': 2275, 'sha256': '020f7f380c8803b3f6a8b1553bb38838d01183dd21b0153d55920b5565e220b7'}}, 'ipsc': {'manifest.json': {'bytes': 1936810, 'sha256': 'eae935f1dca78e3b6c924469e66a1dd05a3da46ae4c8ac913fa059e6aee8aac3'}, 'per_target.csv': {'bytes': 1118175, 'sha256': '0be1dc8d8b3f8b8744fe7ae0b16b3146857454e300cfc40fa2523dcc6fbce96c'}, 'summary.json': {'bytes': 3207, 'sha256': '3b5ea22a76283739d6ce466a9e5becabb05534edc0cdeee863bd2134a0d18a04'}}, 'rpe1': {'manifest.json': {'bytes': 2334067, 'sha256': '84d4c40e15ca208cb62c41c20bd272089215e8f590bf1523e61deebe4ae4127f'}, 'per_target.csv': {'bytes': 352914, 'sha256': '3acf247e534d6e0093f9d6b12b052d36cebca65a97347c2de7d55d8ad7f46148'}, 'summary.json': {'bytes': 1310, 'sha256': 'b6dae1d5292b067dfe9798a4891052aa46fa5fe7e4da4fd12e5a96fb23b4c1f3'}}}, 'gpu': False, 'internet': False, 'training': False, 'new_data_upload': False, 'seed0_original_verdict': {'not_vcc_score': True, 'families': {}, 'contrasts': {'transfer': {'macro_family_delta': 0.0022223326402478804, 'ci95': [0.0005560481898238739, 0.003906613105430528], 'per_family': {'cd4': 0.005157371167645142, 'ipsc': 0.003278141307893017, 'k562': -0.007228983202870191, 'orion': -0.001312912793542074, 'rpe1': 0.011218046722113507}}, 'blind': {'macro_family_delta': -0.00041610608284409645, 'ci95': [-0.001044437734425962, 0.00021331169989399724], 'per_family': {'cd4': 0.0008892897912589716, 'ipsc': -0.0012027071102413604, 'k562': -0.003101047782126553, 'orion': -4.204378669275715e-05, 'rpe1': 0.001375978473581217}}, 'swap': {'macro_family_delta': -0.00022283206947162527, 'ci95': [-0.0004375993762230925, -1.6792033594259923e-05], 'per_family': {'cd4': -0.000211492987606001, 'ipsc': -0.0006714265329419484, 'k562': -1.1466487279845287e-05, 'orion': -0.00017008622798434528, 'rpe1': -4.968811154598633e-05}}, 'prior_permuted': {'macro_family_delta': -0.00023187785388127918, 'ci95': [-0.0005034010875326172, 3.095951565557568e-05], 'per_family': {'cd4': 0.00018728595890411026, 'ipsc': -0.0001184870352250508, 'k562': -0.0005631319308545378, 'orion': 9.173189823874676e-05, 'rpe1': -0.0007567881604696643}}, 'null': {'macro_family_delta': 0.13765651755136984, 'ci95': [0.12968987292583983, 0.14528589137924003], 'per_family': {'cd4': 0.06494299881767775, 'ipsc': 0.10231355491682975, 'k562': 0.18883520874103066, 'orion': 0.04819269049657534, 'rpe1': 0.2839981347847358}}}, 'regime': 'C', 'training_seed': 0, 'bootstrap_scope': 'targets within each observed context; equal families then contexts, conditional on these five families', 'eligible_for_cell_scorer': False, 'evidence_for_context_use': False, 'robust_across_seeds': False, 'next_requirement': 'A separate seed and an independent cell-level benchmark; this readout never authorizes submission.', 'reader_provenance': {'original_reader': '95f0f37610778d26b09806cfed5614d054c8f6fa181a6844480f5277f47c8f22', 'corrected_reader': '21b936b387ec734e0aaeb4b85f982f127a465a84fb3a77dc89917190881a8243', 'amendment': 'eb318d4e79a73ade2d72e489be2e7a65bc37ee0f9210f71aa500a3d0497bec70', 'guard': 'e90e095c25f7e4e17c0cc0e4cfad71f87b57eababf49765db9c5ec624ff0e3b4'}}, 'changes_primary_gate': False, 'readout_sha256': '9b5750f258effa11f994410fddca8fc1ad9a437923da0a48d22dc9a0e79f02e0'})
